# Tableau enrichi — Application des modèles IA

**Projet** : Gallica Images — Classification gravures sur bois vs cuivre  
**Date**   : Avril 2026

Ce notebook charge le CSV brut produit par `collecte_similarite.ipynb`,
applique les modèles ResNet50 v1/v2/v3 sur les 1840 illustrations,
et génère le tableau HTML enrichi avec les colonnes technique IA.

**Entrée**  : `resultats/csv/salomon_segmente.csv`  
             `modeles/bois_cuivre/resnet50_v1|v2|v3.pth`  
**Sortie**  : `resultats/csv/salomon_enrichi_segmente.csv`  
             `resultats/similarite/tableau_salomon_enrichi_segmente.html`

**Pour ajouter une nouvelle version** : ajouter son chemin dans `MODELES`
et relancer — les colonnes s'ajoutent automatiquement au tableau.

⚠️ **Note mémoire** : Ne pas charger YOLO avant ce notebook.

---

## 1. Configuration

**Seule cellule à modifier** pour ajouter ou retirer une version de modèle.

In [1]:
# ── Versions à appliquer — ajouter/enlever selon les besoins ─
MODELES = {
    "v1": "../../../modeles/bois_cuivre/resnet50_v1.pth",
    "v2": "../../../modeles/bois_cuivre/resnet50_v2.pth",
}

CHEMIN_CSV_BRUT   = "../../../resultats/csv/salomon_segmente.csv"
CHEMIN_CSV_ENRICH = "../../../resultats/csv/salomon_enrichi_segmente.csv"
CHEMIN_HTML       = "../../../resultats/similarite/tableau_salomon_enrichi_segmente.html"
# ─────────────────────────────────────────────────────────────

import sys
sys.path.insert(0, "..")
from gallica_utils import charger_resnet, appliquer_modele_df, generer_tableau_html, DEVICE

import pandas as pd
import os

device = DEVICE
print(f"Device  : {device}")
print(f"Modèles : {list(MODELES.keys())}")

Device  : cuda
Modèles : ['v1', 'v2']


## 2. Chargement du CSV brut

In [2]:
df = pd.read_csv(CHEMIN_CSV_BRUT)
print(f"✓ CSV chargé : {df.shape[0]} lignes")
df.head(3)

✓ CSV chargé : 8300 lignes


,salomon_page,salomon_ark,score,titre,auteur,date,corpus,technique,categorie,genre,palette,chromatic_mode,link,result_ark
0,1,bfkfk930z5q,0.9953,[Illustrations de La Métamorphose d'Ovide figu...,Ovide (0043 av. J.-C.-0017). Auteur du texte,1557,NaN,estampe,Ornement typographique,Représentations humaines / Portraits,"blanc, gris, noir",nb,https://openapi.bnf.fr/iiif/image/v3/ark:/1214...,bfkfk930z5q
1,1,bfkfk930z5q,0.9510,Argomento della Galleria Farnese dipinta da An...,"Carracci, Annibale (1560-1609). Peintre",1657,NaN,estampe,Jeu,Représentations humaines / Figures,"blanc, gris, noir",nb,https://openapi.bnf.fr/iiif/image/v3/ark:/1214...,bfkfk84x792
2,1,bfkfk930z5q,0.9507,Les galeries publiques de l'Europe. T1 / par J...,"Armengaud, Jean Germain Désiré (1797-1869). Au...",[1859 TO 1,NaN,estampe,Illustration scientifique / Général,NaN,"noir, blanc, gris",nb,https://openapi.bnf.fr/iiif/image/v3/ark:/1214...,bfkfkb82p56


## 3. Application des modèles

Pour chaque version définie dans `MODELES`, on charge le modèle,
on prédit la technique de toutes les illustrations et on ajoute
deux colonnes `technique_ia_vN` et `technique_ia_conf_vN`.

In [3]:
for version, chemin_pth in MODELES.items():
    col_classe = f"technique_ia_{version}"
    col_conf   = f"technique_ia_conf_{version}"

    # Vérifier si la colonne existe déjà — ne pas recalculer
    if col_classe in df.columns:
        print(f"  {version} — déjà présent, ignoré")
        continue

    if not os.path.exists(chemin_pth):
        print(f"  {version} — modèle introuvable : {chemin_pth}")
        continue

    print(f"\n── Application modèle {version} ──")
    modele = charger_resnet(chemin_pth, device)
    df     = appliquer_modele_df(df, modele, col_classe, col_conf, device)

    # Libérer la mémoire GPU avant le prochain modèle
    import gc, torch
    del modele
    torch.cuda.empty_cache()
    gc.collect()

print("\n✓ Tous les modèles appliqués")


── Application modèle v1 ──
✓ ResNet50 chargé : ../../../modeles/bois_cuivre/resnet50_v1.pth
  8300/8300...
✓ technique_ia_v1 — {'bois': 6621, 'cuivre': 1666, 'inconnu': 13}

── Application modèle v2 ──
✓ ResNet50 chargé : ../../../modeles/bois_cuivre/resnet50_v2.pth
  8300/8300...
✓ technique_ia_v2 — {'bois': 6689, 'cuivre': 1602, 'inconnu': 9}

✓ Tous les modèles appliqués


## 4. Comparaison des versions

In [4]:
print("Résumé des prédictions par version :\n")
for version in MODELES.keys():
    col = f"technique_ia_{version}"
    if col in df.columns:
        print(f"  {version} : {df[col].value_counts().to_dict()}")

# Accord entre toutes les versions pour cuivre
versions_disponibles = [v for v in MODELES.keys()
                        if f"technique_ia_{v}" in df.columns]

if len(versions_disponibles) >= 2:
    print(f"\nAccord toutes versions — cuivre certain :")
    masque = (df[f"technique_ia_{versions_disponibles[0]}"] == "cuivre")
    for v in versions_disponibles[1:]:
        masque = masque & (df[f"technique_ia_{v}"] == "cuivre")
    print(f"  {masque.sum()} illustrations classées cuivre par toutes les versions")

Résumé des prédictions par version :

  v1 : {'bois': 6621, 'cuivre': 1666, 'inconnu': 13}
  v2 : {'bois': 6689, 'cuivre': 1602, 'inconnu': 9}

Accord toutes versions — cuivre certain :
  1296 illustrations classées cuivre par toutes les versions


## 5. Sauvegarde du CSV enrichi

In [5]:
df.to_csv(CHEMIN_CSV_ENRICH, index=False)
print(f"✓ CSV enrichi sauvegardé : {CHEMIN_CSV_ENRICH}")
print(f"  Colonnes : {df.columns.tolist()}")

✓ CSV enrichi sauvegardé : ../../../resultats/csv/salomon_enrichi_segmente.csv
  Colonnes : ['salomon_page', 'salomon_ark', 'score', 'titre', 'auteur', 'date', 'corpus', 'technique', 'categorie', 'genre', 'palette', 'chromatic_mode', 'link', 'result_ark', 'technique_ia_v1', 'technique_ia_conf_v1', 'technique_ia_v2', 'technique_ia_conf_v2']


## 6. Génération du tableau HTML enrichi

In [6]:
# Les colonnes IA disponibles sont déduites automatiquement
versions_html = [v for v in MODELES.keys()
                 if f"technique_ia_{v}" in df.columns]

generer_tableau_html(
    df,
    CHEMIN_HTML,
    colonnes_ia=versions_html,
    port=8085
)

Exception in thread Thread-5 (_serveur):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1073, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1010, in run
    self._target(*self._args, **self._kwargs)
  File "/mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/notebooks/utils.py", line 715, in _serveur
    with http.server.HTTPServer(("", port), handler) as httpd:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socketserver.py", line 457, in __init__
    self.server_bind()
  File "/usr/lib/python3.12/http/server.py", line 136, in server_bind
    socketserver.TCPServer.server_bind(self)
  File "/usr/lib/python3.12/socketserver.py", line 473, in server_bind
    self.socket.bind(self.server_address)
OSError: [Errno 98] Address already in use


✓ Tableau généré : ../../../resultats/similarite/tableau_salomon_enrichi_segmente.html
✓ URL : http://localhost:8085/tableau_similarite_salomon_enrichi_segmente.html


gio: http://localhost:8085/tableau_similarite_salomon_enrichi_segmente.html: Operation not supported
